# Banana Leaf Disease Classification Using Deep Learning

**Formative Assessment 2 — ITLML801 Machine Learning**  
**Level 8 Year 4 IT | Academic Year 2026–2027**

## Objective
Build a deep-learning image classifier for four banana-leaf classes:

- Cordana
- Sigatoka
- Pestalotiopsis
- Healthy

## Model approach
This notebook uses **one transfer-learning approach: EfficientNetB0 pretrained on ImageNet**.
The pretrained feature extractor is first trained with a new four-class classification head,
then upper layers are fine-tuned with a small learning rate.

The notebook is designed to satisfy the assessment requirements:
dataset exploration, preprocessing, leakage-safe train/validation/test splitting,
training-only augmentation, architecture display, Adam optimization, early stopping,
accuracy/loss plots, held-out test evaluation, professional confusion matrices,
precision/recall/F1, error analysis, test-image predictions, model saving, and CSV export.

> **Important:** The notebook targets very high performance, but the final test accuracy is
> calculated from the untouched test set and is never fabricated or hard-coded.

In [ ]:
# ============================================================
# CELL 1 — Environment and imports
# ============================================================
import os
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("=" * 65)
print("ENVIRONMENT")
print("=" * 65)
print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))
print("=" * 65)

## 1. Load the dataset

The assignment uses the Banana Leaf Classification Dataset. The next cell supports
the supplied ZIP when it is uploaded to Kaggle, and also supports an extracted
`OriginalSet` directory. If KaggleHub is available and no local dataset is found,
it can download the official dataset.

In [ ]:
# ============================================================
# CELL 2 — Locate the supplied dataset
# ============================================================
CLASS_NAMES = ["cordana", "sigatoka", "pestalotiopsis", "healthy"]
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

CANDIDATE_ROOTS = [
    Path("/kaggle/input"),
    Path("/mnt/data"),
    Path(".")
]

def find_class_dirs(search_root):
    found = {}
    if not search_root.exists():
        return found

    for root, dirs, files in os.walk(search_root):
        folder = Path(root).name.lower()
        if folder in CLASS_NAMES and folder not in found:
            found[folder] = Path(root)
    return found

class_dirs = {}

for root in CANDIDATE_ROOTS:
    class_dirs.update(find_class_dirs(root))

# If the ZIP was uploaded to Kaggle rather than extracted, extract it.
if len(class_dirs) < 4:
    zip_candidates = []
    for root in CANDIDATE_ROOTS:
        if root.exists():
            zip_candidates.extend(root.rglob("archive (1).zip"))
            zip_candidates.extend(root.rglob("*.zip"))

    if zip_candidates:
        zip_path = zip_candidates[0]
        extract_root = Path("/kaggle/working/banana_leaf_dataset")
        extract_root.mkdir(parents=True, exist_ok=True)

        import zipfile
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(extract_root)

        class_dirs.update(find_class_dirs(extract_root))

# Final fallback: KaggleHub
if len(class_dirs) < 4:
    try:
        import kagglehub
        downloaded = Path(
            kagglehub.dataset_download(
                "ntambaraetienne/banana-leaf-diseases-classification-dataset"
            )
        )
        class_dirs.update(find_class_dirs(downloaded))
    except Exception as exc:
        print("KaggleHub download was not available:", exc)

missing = [c for c in CLASS_NAMES if c not in class_dirs]

if missing:
    raise FileNotFoundError(
        "Could not locate all four class folders. Missing: "
        + ", ".join(missing)
        + "\nUpload/extract the supplied dataset ZIP or enable internet for KaggleHub."
    )

print("Dataset class folders:")
for c in CLASS_NAMES:
    print(f"{c:20s} -> {class_dirs[c]}")

In [ ]:
# ============================================================
# CELL 3 — Build image manifest
# ============================================================
records = []

for label, class_name in enumerate(CLASS_NAMES):
    class_root = class_dirs[class_name]

    for root, _, files in os.walk(class_root):
        for filename in files:
            if Path(filename).suffix.lower() in IMAGE_EXTENSIONS:
                records.append({
                    "filepath": str(Path(root) / filename),
                    "filename": filename,
                    "label": label,
                    "class": class_name
                })

data = pd.DataFrame(records)

if data.empty:
    raise RuntimeError("No image files were found.")

data = data.drop_duplicates(subset="filepath").reset_index(drop=True)

print(f"Total images: {len(data):,}")
display(data.head())

counts = data["class"].value_counts().reindex(CLASS_NAMES)
print("\nImages per class:")
display(counts.to_frame("image_count"))

In [ ]:
# ============================================================
# CELL 4 — Dataset distribution
# ============================================================
plt.figure(figsize=(10, 6))

ax = sns.countplot(
    data=data,
    x="class",
    order=CLASS_NAMES
)

plt.title(
    "Banana Leaf Disease Dataset — Class Distribution",
    fontsize=16,
    fontweight="bold"
)
plt.xlabel("Class")
plt.ylabel("Number of Images")

for container in ax.containers:
    ax.bar_label(container, fontsize=10)

plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 5 — Labeled samples from all four classes
# ============================================================
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

for i, class_name in enumerate(CLASS_NAMES):
    sample = data[data["class"] == class_name].sample(
        1, random_state=SEED
    ).iloc[0]

    image = tf.keras.utils.load_img(
        sample["filepath"],
        target_size=(224, 224)
    )

    axes[i].imshow(image)
    axes[i].set_title(
        class_name.capitalize(),
        fontsize=13,
        fontweight="bold"
    )
    axes[i].axis("off")

plt.suptitle(
    "Representative Images From All Four Classes",
    fontsize=17,
    fontweight="bold"
)
plt.tight_layout()
plt.show()

## 2. Preprocess and split the data

The data is divided into 70% training, 15% validation, and 15% testing using
stratification. The test set is kept untouched until final evaluation.

In [ ]:
# ============================================================
# CELL 6 — Leakage-safe stratified split
# ============================================================
train_df, temp_df = train_test_split(
    data,
    test_size=0.30,
    random_state=SEED,
    stratify=data["label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Dataset split sizes")
print("-" * 35)
print(f"Training:   {len(train_df):4d} ({len(train_df)/len(data):.1%})")
print(f"Validation: {len(val_df):4d} ({len(val_df)/len(data):.1%})")
print(f"Test:       {len(test_df):4d} ({len(test_df)/len(data):.1%})")
print(f"Total:      {len(data):4d}")

In [ ]:
# ============================================================
# CELL 7 — Verify no image appears in more than one split
# ============================================================
train_paths = set(train_df["filepath"])
val_paths = set(val_df["filepath"])
test_paths = set(test_df["filepath"])

overlap_train_val = len(train_paths & val_paths)
overlap_train_test = len(train_paths & test_paths)
overlap_val_test = len(val_paths & test_paths)

print("Train ∩ Validation:", overlap_train_val)
print("Train ∩ Test:      ", overlap_train_test)
print("Validation ∩ Test: ", overlap_val_test)

assert overlap_train_val == 0
assert overlap_train_test == 0
assert overlap_val_test == 0

print("\n✓ Data leakage check passed.")

In [ ]:
# ============================================================
# CELL 8 — Class distribution in each split
# ============================================================
split_table = pd.DataFrame({
    "Train": train_df["class"].value_counts(),
    "Validation": val_df["class"].value_counts(),
    "Test": test_df["class"].value_counts()
}).reindex(CLASS_NAMES)

display(split_table)

split_table.plot(
    kind="bar",
    figsize=(11, 6)
)

plt.title(
    "Class Distribution Across Train, Validation and Test",
    fontsize=16,
    fontweight="bold"
)
plt.xlabel("Class")
plt.ylabel("Number of Images")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 9 — TensorFlow input pipeline
# ============================================================
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 4
AUTOTUNE = tf.data.AUTOTUNE

def load_image(filepath, label):
    image = tf.io.read_file(filepath)
    image = tf.image.decode_image(
        image,
        channels=3,
        expand_animations=False
    )
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32)
    return image, label

def make_dataset(frame, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices(
        (
            frame["filepath"].values,
            frame["label"].values
        )
    )

    if shuffle:
        ds = ds.shuffle(
            buffer_size=len(frame),
            seed=SEED,
            reshuffle_each_iteration=True
        )

    ds = ds.map(
        load_image,
        num_parallel_calls=AUTOTUNE
    )
    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(AUTOTUNE)

    return ds

train_ds = make_dataset(train_df, shuffle=True)
val_ds = make_dataset(val_df, shuffle=False)
test_ds = make_dataset(test_df, shuffle=False)

print("TensorFlow datasets created successfully.")

In [ ]:
# ============================================================
# CELL 10 — Training-only augmentation
# ============================================================
data_augmentation = keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.08),
        layers.RandomZoom(0.10),
        layers.RandomTranslation(0.05, 0.05),
        layers.RandomContrast(0.10)
    ],
    name="training_augmentation"
)

print("Augmentation is configured inside the model and is active only during training.")

In [ ]:
# ============================================================
# CELL 11 — Visualize augmentation
# ============================================================
images, labels = next(iter(train_ds))

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes = axes.ravel()

for i in range(8):
    augmented = data_augmentation(
        tf.expand_dims(images[i], 0),
        training=True
    )[0]

    axes[i].imshow(
        tf.cast(tf.clip_by_value(augmented, 0, 255), tf.uint8)
    )
    axes[i].set_title(
        CLASS_NAMES[int(labels[i])].capitalize()
    )
    axes[i].axis("off")

plt.suptitle(
    "Examples of Training-Only Augmentation",
    fontsize=16,
    fontweight="bold"
)
plt.tight_layout()
plt.show()

## 3. Build the model — Transfer Learning

EfficientNetB0 pretrained on ImageNet is used as the single model approach.
Global average pooling replaces a large flattening layer, reducing unnecessary
parameters. A four-neuron softmax output produces probabilities for the four
required disease classes.

In [ ]:
# ============================================================
# CELL 12 — Build EfficientNetB0 transfer-learning model
# ============================================================
base_model = tf.keras.applications.EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(*IMG_SIZE, 3)
)

base_model.trainable = False

inputs = layers.Input(
    shape=(*IMG_SIZE, 3),
    name="input_image"
)

x = data_augmentation(inputs)

# EfficientNetB0 includes its own input rescaling.
x = base_model(
    x,
    training=False
)

x = layers.GlobalAveragePooling2D(
    name="global_average_pooling"
)(x)

x = layers.BatchNormalization(
    name="head_batch_norm"
)(x)

x = layers.Dropout(
    0.35,
    name="head_dropout"
)(x)

outputs = layers.Dense(
    NUM_CLASSES,
    activation="softmax",
    name="predictions"
)(x)

model = keras.Model(
    inputs=inputs,
    outputs=outputs,
    name="BananaLeaf_EfficientNetB0"
)

model.summary()

In [ ]:
# ============================================================
# CELL 13 — Class weights
# ============================================================
class_weight_values = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(NUM_CLASSES),
    y=train_df["label"].values
)

class_weights = {
    int(class_id): float(weight)
    for class_id, weight in enumerate(class_weight_values)
}

print("Class weights:")
for class_id, weight in class_weights.items():
    print(f"{CLASS_NAMES[class_id]:18s}: {weight:.4f}")

### Architecture explanation

- **Input:** 224×224 RGB image.
- **Augmentation:** applied only during training.
- **EfficientNetB0:** pretrained ImageNet feature extractor.
- **GlobalAveragePooling2D:** converts spatial feature maps into a compact feature vector.
- **Batch normalization + dropout:** improves stability and helps reduce overfitting.
- **Dense(4, softmax):** outputs probabilities for Cordana, Sigatoka, Pestalotiopsis,
  and Healthy.

In [ ]:
# ============================================================
# CELL 14 — Stage 1: train classification head
# ============================================================
model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=1e-3
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

stage1_callbacks = [
    keras.callbacks.ModelCheckpoint(
        "best_banana_leaf_model.keras",
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
        verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

history_head = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    class_weight=class_weights,
    callbacks=stage1_callbacks,
    verbose=1
)

In [ ]:
# ============================================================
# CELL 15 — Stage 2: controlled fine-tuning
# ============================================================
base_model.trainable = True

# Freeze the earliest feature-extraction layers.
# Keep BatchNormalization frozen for stable fine-tuning.
for layer in base_model.layers[:150]:
    layer.trainable = False

for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

trainable_layers = sum(
    int(layer.trainable)
    for layer in base_model.layers
)

print("Trainable EfficientNetB0 layers:", trainable_layers)

In [ ]:
# ============================================================
# CELL 16 — Recompile with a small learning rate
# ============================================================
model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=1e-5
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    class_weight=class_weights,
    callbacks=[
        keras.callbacks.ModelCheckpoint(
            "best_banana_leaf_model.keras",
            monitor="val_accuracy",
            mode="max",
            save_best_only=True,
            verbose=1
        ),
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=5,
            restore_best_weights=True,
            verbose=1
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.3,
            patience=2,
            min_lr=1e-8,
            verbose=1
        )
    ],
    verbose=1
)

In [ ]:
# ============================================================
# CELL 17 — Combine training histories
# ============================================================
def get_history(history_obj, key):
    return history_obj.history.get(key, [])

train_acc = get_history(history_head, "accuracy") + get_history(history_fine, "accuracy")
val_acc = get_history(history_head, "val_accuracy") + get_history(history_fine, "val_accuracy")
train_loss = get_history(history_head, "loss") + get_history(history_fine, "loss")
val_loss = get_history(history_head, "val_loss") + get_history(history_fine, "val_loss")

epochs = np.arange(1, len(train_acc) + 1)

best_epoch = int(np.argmax(val_acc) + 1)
best_val_accuracy = float(np.max(val_acc))

print("Completed epochs:", len(epochs))
print(f"Best validation accuracy: {best_val_accuracy * 100:.2f}%")
print(f"Best epoch: {best_epoch}")

In [ ]:
# ============================================================
# CELL 18 — Professional accuracy curve
# ============================================================
plt.figure(figsize=(12, 6))

plt.plot(
    epochs,
    train_acc,
    marker="o",
    linewidth=2,
    label="Training Accuracy"
)

plt.plot(
    epochs,
    val_acc,
    marker="o",
    linewidth=2,
    label="Validation Accuracy"
)

plt.axhline(
    0.98,
    linestyle="--",
    linewidth=1.8,
    label="98% Reference"
)

plt.title(
    "Training and Validation Accuracy",
    fontsize=17,
    fontweight="bold"
)
plt.xlabel("Epoch", fontsize=13)
plt.ylabel("Accuracy", fontsize=13)
plt.ylim(0, 1.05)
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 19 — Professional loss curve
# ============================================================
plt.figure(figsize=(12, 6))

plt.plot(
    epochs,
    train_loss,
    marker="o",
    linewidth=2,
    label="Training Loss"
)

plt.plot(
    epochs,
    val_loss,
    marker="o",
    linewidth=2,
    label="Validation Loss"
)

plt.title(
    "Training and Validation Loss",
    fontsize=17,
    fontweight="bold"
)
plt.xlabel("Epoch", fontsize=13)
plt.ylabel("Loss", fontsize=13)
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

## Training interpretation

A healthy training process should show decreasing training/validation loss and
increasing accuracy. A large and persistent gap between training and validation
accuracy, together with rising validation loss, is evidence of overfitting.
If both accuracies remain low and losses remain high, the model is underfitting.

The final conclusion must be based on the actual curves printed above.

In [ ]:
# ============================================================
# CELL 20 — Final held-out test evaluation
# ============================================================
test_loss, test_accuracy = model.evaluate(
    test_ds,
    verbose=1
)

print("\n" + "=" * 70)
print("FINAL HELD-OUT TEST RESULT")
print("=" * 70)
print(f"Test loss:     {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy * 100:.2f}%")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 21 — Predictions on the untouched test set
# ============================================================
y_true = test_df["label"].to_numpy()

y_probability = model.predict(
    test_ds,
    verbose=1
)

y_pred = np.argmax(
    y_probability,
    axis=1
)

confidence = np.max(
    y_probability,
    axis=1
)

print("Test predictions generated:", len(y_pred))

In [ ]:
# ============================================================
# CELL 22 — Professional count confusion matrix
# ============================================================
cm = confusion_matrix(
    y_true,
    y_pred,
    labels=np.arange(NUM_CLASSES)
)

plt.figure(figsize=(10, 8))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    linewidths=1,
    linecolor="white",
    cbar=True,
    xticklabels=[c.capitalize() for c in CLASS_NAMES],
    yticklabels=[c.capitalize() for c in CLASS_NAMES],
    annot_kws={"fontsize": 14, "fontweight": "bold"}
)

plt.title(
    "Confusion Matrix — Banana Leaf Disease Classification",
    fontsize=18,
    fontweight="bold",
    pad=18
)
plt.xlabel("Predicted Class", fontsize=14, fontweight="bold")
plt.ylabel("True Class", fontsize=14, fontweight="bold")
plt.xticks(rotation=25, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 23 — Professional normalized confusion matrix
# ============================================================
cm_norm = confusion_matrix(
    y_true,
    y_pred,
    labels=np.arange(NUM_CLASSES),
    normalize="true"
)

plt.figure(figsize=(10, 8))

sns.heatmap(
    cm_norm,
    annot=True,
    fmt=".1%",
    cmap="Blues",
    vmin=0,
    vmax=1,
    linewidths=1,
    linecolor="white",
    cbar=True,
    xticklabels=[c.capitalize() for c in CLASS_NAMES],
    yticklabels=[c.capitalize() for c in CLASS_NAMES],
    annot_kws={"fontsize": 13, "fontweight": "bold"}
)

plt.title(
    "Normalized Confusion Matrix",
    fontsize=18,
    fontweight="bold",
    pad=18
)
plt.xlabel("Predicted Class", fontsize=14, fontweight="bold")
plt.ylabel("True Class", fontsize=14, fontweight="bold")
plt.xticks(rotation=25, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 24 — Classification report
# ============================================================
target_names = [c.capitalize() for c in CLASS_NAMES]

report_text = classification_report(
    y_true,
    y_pred,
    labels=np.arange(NUM_CLASSES),
    target_names=target_names,
    digits=4,
    zero_division=0
)

print("CLASSIFICATION REPORT")
print("=" * 75)
print(report_text)

In [ ]:
# ============================================================
# CELL 25 — Per-class precision, recall and F1 visualization
# ============================================================
report_dict = classification_report(
    y_true,
    y_pred,
    labels=np.arange(NUM_CLASSES),
    target_names=target_names,
    output_dict=True,
    zero_division=0
)

metrics_df = pd.DataFrame({
    "Precision": [report_dict[c]["precision"] for c in target_names],
    "Recall": [report_dict[c]["recall"] for c in target_names],
    "F1-Score": [report_dict[c]["f1-score"] for c in target_names]
}, index=target_names)

display(metrics_df.round(4))

ax = metrics_df.plot(
    kind="bar",
    figsize=(12, 7)
)

ax.set_title(
    "Per-Class Precision, Recall and F1-Score",
    fontsize=17,
    fontweight="bold"
)
ax.set_xlabel("Class")
ax.set_ylabel("Score")
ax.set_ylim(0, 1.05)
ax.grid(axis="y", alpha=0.25)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 26 — Error analysis
# ============================================================
results = test_df.copy()

results["true_class"] = [
    CLASS_NAMES[i] for i in y_true
]

results["predicted_class"] = [
    CLASS_NAMES[i] for i in y_pred
]

results["confidence"] = confidence

incorrect = results[
    results["label"] != y_pred
].copy()

print(f"Incorrect predictions: {len(incorrect)}")
print(f"Test error rate: {len(incorrect) / len(results) * 100:.2f}%")

if len(incorrect):
    print("\nMost confident incorrect predictions:")
    display(
        incorrect.sort_values(
            "confidence",
            ascending=False
        )[
            [
                "filename",
                "true_class",
                "predicted_class",
                "confidence"
            ]
        ].head(15)
    )
else:
    print("No incorrect test predictions.")

In [ ]:
# ============================================================
# CELL 27 — Display test images with true/predicted classes
# ============================================================
def show_prediction_grid(frame, title, n=12):
    n = min(n, len(frame))
    if n == 0:
        print("No images to display.")
        return

    subset = frame.sample(
        n,
        random_state=SEED
    )

    cols = 4
    rows = int(np.ceil(n / cols))

    fig, axes = plt.subplots(
        rows,
        cols,
        figsize=(16, 4 * rows)
    )

    axes = np.array(axes).reshape(-1)

    for ax, (_, row) in zip(axes, subset.iterrows()):
        image = tf.keras.utils.load_img(
            row["filepath"],
            target_size=IMG_SIZE
        )

        ax.imshow(image)

        ax.set_title(
            f"True: {row['true_class'].capitalize()}\n"
            f"Pred: {row['predicted_class'].capitalize()}\n"
            f"Confidence: {row['confidence']:.1%}",
            fontsize=10
        )
        ax.axis("off")

    for ax in axes[n:]:
        ax.axis("off")

    plt.suptitle(
        title,
        fontsize=18,
        fontweight="bold"
    )
    plt.tight_layout()
    plt.show()

show_prediction_grid(
    results,
    "Held-Out Test Images — True vs Predicted Class",
    n=12
)

In [ ]:
# ============================================================
# CELL 28 — Display misclassified test images
# ============================================================
show_prediction_grid(
    incorrect,
    "Misclassified Test Images — Error Analysis",
    n=12
)

## Error analysis and model conclusion

The confusion matrix identifies the classes that are most frequently confused.
The classification report provides class-level precision, recall and F1-score.

Use the actual curves and test metrics to decide whether the model is:
- **well fitted:** training and validation performance are both strong and reasonably close;
- **overfitting:** training performance is substantially higher while validation performance
  deteriorates;
- **underfitting:** both training and validation performance remain weak.

Do not claim 98–100% accuracy unless the held-out test evaluation above actually produces it.

In [ ]:
# ============================================================
# CELL 29 — Save trained model
# ============================================================
MODEL_PATH = "/kaggle/working/banana_leaf_disease_efficientnetb0.keras"

model.save(MODEL_PATH)

print("Trained model saved successfully:")
print(MODEL_PATH)

In [ ]:
# ============================================================
# CELL 30 — Save prediction CSV
# ============================================================
PREDICTION_CSV = "/kaggle/working/banana_leaf_test_predictions.csv"

prediction_df = pd.DataFrame({
    "filename": test_df["filename"].values,
    "true_class": [CLASS_NAMES[i] for i in y_true],
    "predicted_class": [CLASS_NAMES[i] for i in y_pred],
    "confidence": confidence
})

prediction_df.to_csv(
    PREDICTION_CSV,
    index=False
)

print("Prediction CSV saved successfully:")
print(PREDICTION_CSV)

display(prediction_df.head(10))

In [ ]:
# ============================================================
# CELL 31 — Final results dashboard
# ============================================================
print("\n" + "=" * 75)
print("BANANA LEAF DISEASE CLASSIFICATION — FINAL RESULTS")
print("=" * 75)

print(f"Model:                  EfficientNetB0 Transfer Learning")
print(f"Classes:                {', '.join(CLASS_NAMES)}")
print(f"Training images:        {len(train_df)}")
print(f"Validation images:      {len(val_df)}")
print(f"Test images:            {len(test_df)}")
print(f"Best validation acc.:   {best_val_accuracy * 100:.2f}%")
print(f"Final test loss:        {test_loss:.4f}")
print(f"Final test accuracy:    {test_accuracy * 100:.2f}%")
print(f"Model file:             {MODEL_PATH}")
print(f"Prediction CSV:         {PREDICTION_CSV}")
print("=" * 75)

if test_accuracy >= 0.98:
    print("✓ The genuine held-out test accuracy reached the 98% target.")
else:
    print("The 98% target was not reached by this run.")
    print("Report the genuine test result and use the confusion matrix/error analysis.")

# Final submission checklist

Before submission:

1. Run the entire notebook from the first cell to the last cell.
2. Confirm every cell has its output visible.
3. Confirm the leakage checks are all zero.
4. Confirm augmentation is used only through the training path.
5. Confirm the held-out test set is evaluated only at the final evaluation stage.
6. Confirm the confusion matrix contains all four class names.
7. Confirm the classification report contains precision, recall and F1-score.
8. Confirm the trained model is saved.
9. Confirm the prediction CSV is saved.
10. Rename the notebook to the required submission format:
   **Names_RegNo_FA2**

The assignment states that the notebook should contain code, outputs and explanations,
and that the saved model and prediction CSV should be produced by the notebook.